<h1> Experiment anti HOM probability against bandwidth </h1>

In [ ]:
#General imports
resol = 300
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FuncFormatter
import pandas as pd

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})

import sys
import numpy as np
from pathlib import Path
pi = np.pi

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))

#Local imports
from experiments.coincidence_vs_bandwidth import run_coincidence_vs_bandwidth
from src.coincidence_theory import get_C

<h2> Execute the experiment </h2>

In [ ]:
#Physical parameters
omega_A = 10*pi
gamma_A = 5*pi

#Different photon frequencies
omega_q_tab = [9.5*pi, 9*pi, 8.5*pi]

sigma_w = 0.5*pi
monochr = False

#Prepare a tab of IR and UV cutoffs for each experiment
nb_pts_bandwidth = 15
ir_tab_list = []
uv_tab_list = []

#Experiments 1,2,3,4 : different central frequencies
central_freqs_1 = [10*pi, 9.5*pi, 9*pi, 8.5*pi]
for i in range(4):
    bandwidth_tab = np.linspace(pi, 15*pi, nb_pts_bandwidth)
    ir_tab_list.append(central_freqs_1[i] - bandwidth_tab)
    uv_tab_list.append(central_freqs_1[i] + bandwidth_tab)

#Experiments 5,6,7,8 : bad choices of central frequencies, below physical TLS frequency
central_freqs_2 = [0*pi, 0.5*pi, 1*pi, 1.5*pi]
for i in range(4):
    bandwidth_tab = np.linspace(pi, 15*pi, nb_pts_bandwidth) #negative frequencies included
    ir_tab_list.append(central_freqs_2[i] - bandwidth_tab)
    uv_tab_list.append(central_freqs_2[i] + bandwidth_tab)

Run the experiment

In [ ]:
#Order of the bare parameters
n_tab = [-1,1] #to run
index_omega_q_to_run = [1,2,3]
xp_to_run = [2,3,4,5]

for n in n_tab:
        for index_omega_q in index_omega_q_to_run:
                for index_experiment in xp_to_run:
                        print(f"Running index omega q {index_omega_q} for frequency window {index_experiment}, n = {n}")

                        omega_q = omega_q_tab[index_omega_q-1]

                        i = index_experiment - 1

                        ir_tab = ir_tab_list[i]
                        uv_tab = uv_tab_list[i]

                        #Parameters of the simulation
                        L = 50

                        param_cavity_physical = {'omega_A': omega_A, 'gamma_A': gamma_A, 'L': L}

                        param_time_evol = {'T': L/2, 'dt': 0.01}

                        param_photons = {'omega_p': [omega_q, omega_q], 
                                        'sigma_w': [sigma_w, sigma_w],
                                        'x_0': [-L/4, -L/4]}
                                
                        _, _, coincidence_tab = run_coincidence_vs_bandwidth(param_photons, param_cavity_physical, param_time_evol, ir_tab, uv_tab, 
                                                                        index_omega_q, index_experiment, n=n, monochr=monochr)
                        
                        print("------------- \n")

<h2> Convergence per truncation scheme </h2>

In [ ]:
monochr = False

Recover the data

In [ ]:
# =========================
# Common parameters
# =========================
if not monochr:
    raise ValueError("Wrong cell: set monochr = True to plot monochromatic data")

index_experiments = [5, 4, 3, 2]  # Experiments to plot
max_x = 16*pi

n_value = 1
csv_root = Path("../results/csv_files/coincidence_vs_bandwidth")
colors = ["#ed9a00", "#8e3e7a", "#003f5c"]
labels = [r'$\omega_q = 9.5\pi$', r'$\omega_q = 9\pi$', r'$\omega_q = 8.5\pi$']
subfig_labels = [r'(a) $\omega_{\rm ref} = 0\pi$', 
                 r'(b) $\omega_{\rm ref} = 8.5\pi$',
                 r'(c) $\omega_{\rm ref} = 9\pi$',
                 r'(d) $\omega_{\rm ref} = 9.5\pi$',]

# =========================
# Prepare figure and axes (1 row)
# =========================
fig, axs = plt.subplots(1, 4, figsize=(16, 4), dpi=300, sharex=True)
axs = axs.flat
fig.subplots_adjust(wspace=0.3)

# =========================
# Loop over experiments
# =========================
for idx, ax in enumerate(axs):
    index_experiment = index_experiments[idx]

    coincidence_to_plot = []
    omega_available = []
    bandwidth_to_plot = []
    theoretical_val_list = []
    relative_error_list = []

    # Load data
    for j in range(len(omega_q_tab)):
        index_omega_q = j + 1
        data_file = csv_root / "monochr" / f"{index_experiment}" / f"coincidence_vs_bandwidth_omega{index_omega_q}_xp{index_experiment}_n{n_value}.csv"

        try:
            df = pd.read_csv(data_file)
            coincidence_tab = df['coincidence_tab'].to_numpy()
            ir_tab = df['ir_tab'].to_numpy()
            uv_tab = df['uv_tab'].to_numpy()

            theoretical_val = get_C(omega_A, gamma_A, omega_q_tab[index_omega_q-1])

            coincidence_to_plot.append(coincidence_tab)
            omega_available.append(j)
            bandwidth_to_plot.append(0.5*(uv_tab - ir_tab))
            theoretical_val_list.append(theoretical_val)
            relative_error_list.append(np.abs(coincidence_to_plot[-1][-1] - theoretical_val_list[-1])/theoretical_val_list[-1])

        except FileNotFoundError:
            print(f'xp{index_experiment}, omega_q index {index_omega_q} not available: {data_file}')
            continue

    # =========================
    # Plots on current axis
    # =========================
    for j, omega_idx in enumerate(omega_available):
        ax.plot(
            bandwidth_to_plot[j],
            coincidence_to_plot[j],
            marker='o',
            markersize=2,
            label=labels[omega_idx],
            color=colors[omega_idx],
            linewidth=1,
            alpha=0.95,
            zorder=3
        )

        ax.hlines(
            theoretical_val_list[j],
            0,
            max_x,
            color=colors[omega_idx],
            linestyle='--',
            linewidth=0.8,
            alpha=0.8,
            zorder=2
        )

        ax.fill_between(
            x=np.linspace(0, max_x, 500),
            y1=0.95*theoretical_val_list[j],
            y2=1.05*theoretical_val_list[j],
            color=colors[omega_idx],
            alpha=0.2,
            zorder=1
        )

        if idx == 3 and (j == 1 or j == 2):
            y_text = 1.05*coincidence_to_plot[j][-1] + 0.05
        else:
            y_text = 1.05*theoretical_val_list[j] + 0.05

        ax.text(
            0.7,
            y_text,
            rf'RE = {relative_error_list[j]*100:.2f}\%',
            color=colors[omega_idx],
            fontsize=10,
            va='center',
            ha='left',
            transform=ax.get_yaxis_transform()
        )

    # =========================
    # Panel label + omega_ref
    # =========================
    ax.text(0.02, 1.05, subfig_labels[idx], 
            transform=ax.transAxes, fontsize=20, fontweight='bold', va='bottom')

    ax.grid(color='0.9', linestyle='-', linewidth=0.5)
    ax.set_xlabel(r'\textbf{Bandwidth} $\Lambda$', fontsize=11)
    ax.set_ylabel(r'\textbf{Coincidence} $\mathcal{C}$', fontsize=11)
    ax.tick_params(axis='both', which='major', length=4, width=0.8, labelsize=9)
    ax.set_xlim([0, max_x])
    ax.set_ylim([0, 1.1])

    ax.xaxis.set_major_locator(MultipleLocator(2*np.pi))
    def pi_formatter(x, pos):
        """Format an axis value in units of pi."""
        n = x / np.pi
        if np.isclose(n, 0):
            return "0"
        elif np.isclose(n, 1):
            return r"$\pi$"
        else:
            return rf"${int(round(n))}\pi$"
    ax.xaxis.set_major_formatter(FuncFormatter(pi_formatter))

    for item in [ax.xaxis.label, ax.yaxis.label]:
        item.set_fontsize(15)

    for item in (ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(15)

# =========================
# Common legend below
# =========================
handles, _ = axs[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0),
    ncol=3,
    frameon=False,
    fontsize=20
)

plt.tight_layout()
#plt.savefig("../results/fig/coincidence_vs_bandwidth_baseline.pdf", bbox_inches='tight', pad_inches=0.2)
plt.show()


Recover the non-monochromatic data


In [ ]:
# =========================
# Non-monochromatic consistency plot
# =========================
if monochr:
    raise ValueError("Wrong cell: set monochr = False to plot non-monochromatic data")

from matplotlib.lines import Line2D

index_experiments = [5, 4, 3, 2]
max_x = 16*pi
csv_root = Path("../results/csv_files/coincidence_vs_bandwidth")

base_colors = ["#ed9a00", "#8e3e7a", "#003f5c"]
baseline_colors = ["#c87300", "#b65496", "#1d6582"]
omega_labels = [r'$\omega_q = 9.5\pi$', r'$\omega_q = 9\pi$', r'$\omega_q = 8.5\pi$']
subfig_labels = [r'(a) $\omega_{\rm ref} = 0\pi$', 
                 r'(b) $\omega_{\rm ref} = 8.5\pi$',
                 r'(c) $\omega_{\rm ref} = 9\pi$',
                 r'(d) $\omega_{\rm ref} = 9.5\pi$',]
curve_styles = {
    1: {"name": "corrected", "marker": "o", "linestyle": "-", "alpha": 0.95, "linewidth": 1.3, "markersize": 2.8},
    -1: {"name": "baseline (no correction)", "marker": "s", "linestyle": "--", "alpha": 0.9, "linewidth": 1.15, "markersize": 2.6},
}

fig, axs = plt.subplots(1, 4, figsize=(16, 4), dpi=300, sharex=True, sharey=True)
axs = axs.flat
fig.subplots_adjust(wspace=0.3)

for idx, ax in enumerate(axs):
    index_experiment = index_experiments[idx]

    for omega_idx, omega_label in enumerate(omega_labels):
        index_omega_q = omega_idx + 1

        for n_value in [1, -1]:
            style = curve_styles[n_value]
            data_file = csv_root / "non_monochr" / f"{index_experiment}" / f"coincidence_vs_bandwidth_omega{index_omega_q}_xp{index_experiment}_n{n_value}.csv"

            try:
                df = pd.read_csv(data_file)
            except FileNotFoundError:
                print(f'xp{index_experiment}, omega_q index {index_omega_q}, n={n_value} not available: {data_file}')
                continue

            coincidence_tab = df['coincidence_tab'].to_numpy()
            ir_tab = df['ir_tab'].to_numpy()
            uv_tab = df['uv_tab'].to_numpy()
            bandwidth_tab = 0.5*(uv_tab - ir_tab)

            color = base_colors[omega_idx] if n_value == 1 else baseline_colors[omega_idx]
            ax.plot(
                bandwidth_tab,
                coincidence_tab,
                marker=style["marker"],
                markersize=style["markersize"],
                label=f'{omega_label} - {style["name"]}',
                color=color,
                linestyle=style["linestyle"],
                linewidth=style["linewidth"],
                alpha=style["alpha"],
                zorder=3 if n_value == 1 else 2,
            )

    ax.text(0.02, 1.05, subfig_labels[idx],
            transform=ax.transAxes, fontsize=20, fontweight='bold', va='bottom')

    ax.grid(color='0.9', linestyle='-', linewidth=0.5)
    ax.set_xlabel(r'\textbf{Bandwidth} $\Lambda$', fontsize=15)
    ax.set_ylabel(r'\textbf{Coincidence} $\mathcal{C}$', fontsize=15)
    ax.tick_params(axis='both', which='major', length=4, width=0.8, labelsize=15)
    ax.set_xlim([0, max_x])
    ax.set_ylim([0, 1.1])

    ax.xaxis.set_major_locator(MultipleLocator(2*np.pi))
    def pi_formatter(x, pos):
        """Format an axis value in units of pi."""
        n = x / np.pi
        if np.isclose(n, 0):
            return "0"
        elif np.isclose(n, 1):
            return r"$\pi$"
        else:
            return rf"${int(round(n))}\pi$"
    ax.xaxis.set_major_formatter(FuncFormatter(pi_formatter))

legend_handles = []
legend_labels = []
for omega_idx, omega_label in enumerate(omega_labels):
    for n_value in [1, -1]:
        style = curve_styles[n_value]
        color = base_colors[omega_idx] if n_value == 1 else baseline_colors[omega_idx]
        legend_handles.append(Line2D(
            [0], [0],
            color=color,
            marker=style["marker"],
            linestyle=style["linestyle"],
            linewidth=style["linewidth"],
            markersize=5,
            alpha=style["alpha"],
        ))
        legend_labels.append(f'{omega_label} - {style["name"]}')

fig.legend(
    legend_handles,
    legend_labels,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.01),
    ncol=3,
    frameon=False,
    fontsize=14,
)

plt.tight_layout()
plt.savefig("../results/fig/coincidence_vs_bandwidth_non_monochr_consistency.pdf", bbox_inches='tight', pad_inches=0.2)
plt.show()
